# Corrected retrain + candidate-structure features

Rebuilds the candidate set as **only intersections below the City's 5-crash screen**
(`crashes_feat < 5`) and refits the frozen XGBoost Tweedie model on three feature sets, all
with genuine 5-fold out-of-fold scoring:

- **A** crash-only
- **D** crash + infrastructure
- **E** crash + infrastructure + **spatial-neighbor** features (local crash pressure,
  proximity to the City's >=5-crash sites, grid density)

Compared to the persistence baseline and the City ($0 here — these are sites it ignores).

### Upload these 6 files (drag into the Files panel, or use the upload cell)
| file | on your machine, in |
|---|---|
| `candidate_panel.parquet` | `data\\model\\verified_run\\` |
| `feature_table.parquet` | `data\\model\\verified_run\\` |
| `oof_scores.parquet` | `data\\model\\verified_run\\` |
| `infra_features.parquet` | `data\\model\\` |
| `spatial_features_verified.parquet` | `data\\model\\` |
| `frozen_params.json` | `data\\model\\` |

Run the cells top to bottom.


In [ ]:
# 1. deps (xgboost usually preinstalled on Colab; this makes sure)
!pip -q install xgboost 2>/dev/null
import xgboost, sklearn, scipy, pandas, numpy
print("xgboost", xgboost.__version__, "| ready")


In [ ]:
# 2. upload the 6 files (pick all at once). They land flat in the working dir.
from google.colab import files
up = files.upload()
need = {"candidate_panel.parquet","feature_table.parquet","oof_scores.parquet",
        "infra_features.parquet","spatial_features_verified.parquet","frozen_params.json"}
missing = need - set(up)
print("\nMISSING:", missing if missing else "none - good to go")


In [ ]:
# 3. corrected retrain: A vs D vs E (crash+infra+spatial), genuine OOF
import json, numpy as np, pandas as pd, xgboost as xgb
from scipy import stats
from sklearn.model_selection import GroupKFold, StratifiedKFold

COST = 5_175_524      # $/KSI event (FHWA-SA-25-021, 24.3% fatal share)
EFF  = 0.30           # treatment effectiveness
PROG = 3_200_000      # top-500 program cost after 90% HSIP
SEED = 42; NF = 5; CITY_MIN = 5

CRASH = ["crashes_36mo","crashes_72mo","ped_crashes_72mo","bike_crashes_72mo","broadside_72mo",
 "left_turn_72mo","dui_72mo","night_72mo","ped_row_violation_72mo","years_since_last_crash",
 "distinct_crash_days_72mo","worst_severity_72mo","crash_trend_slope","emergence_velocity",
 "emergence_acceleration","mann_kendall_tau","changepoint_prob","ewma_crashes","momentum_ratio","covid_period_share"]

panel = pd.read_parquet("candidate_panel.parquet", columns=["intersection_id","KSI_label"])
feats = pd.read_parquet("feature_table.parquet")
infra = pd.read_parquet("infra_features.parquet")
spat  = pd.read_parquet("spatial_features_verified.parquet")   # crashes_feat + 6 neighbor features
oof   = pd.read_parquet("oof_scores.parquet", columns=["intersection_id","spatial_block","persistence_baseline_score"])
fp = json.load(open("frozen_params.json"))["frozen"]
params = dict(objective="reg:tweedie",
    tweedie_variance_power=float(fp["tweedie_variance_power"]), max_depth=int(fp["max_depth"]),
    learning_rate=float(fp["learning_rate"]), n_estimators=int(fp["n_estimators"]),
    reg_alpha=float(fp["reg_alpha"]), reg_lambda=float(fp["reg_lambda"]),
    min_child_weight=int(fp["min_child_weight"]), subsample=float(fp["subsample"]),
    colsample_bytree=float(fp["colsample_bytree"]), random_state=42, verbosity=0)

df = (panel.merge(feats, on="intersection_id", how="left")
           .merge(infra, on="intersection_id", how="left")
           .merge(spat,  on="intersection_id", how="left")
           .merge(oof,   on="intersection_id", how="left"))

# CORRECTED candidate set: below the City screen (crashes_feat from the spatial file)
corr = df[df["crashes_feat"] < CITY_MIN].reset_index(drop=True)
print(f"corrected candidates (<{CITY_MIN} crashes): {len(corr)}  "
      f"| pos>=1={int((corr.KSI_label>=1).sum())}  pos>=2={int((corr.KSI_label>=2).sum())}\n")

infra_cols = [c for c in infra.columns if c != "intersection_id" and corr[c].fillna(0).var() > 0]
spat_cols  = [c for c in spat.columns  if c not in ("intersection_id","crashes_feat") and corr[c].fillna(0).var() > 0]
SETS = {"A_crash_only": CRASH,
        "D_crash_infra": CRASH + infra_cols,
        "E_crash_infra_spatial": CRASH + infra_cols + spat_cols}
print("spatial features used:", spat_cols, "\n")

y = corr.KSI_label.values.astype(float)
groups = corr.spatial_block.values
base = corr.persistence_baseline_score.values

def folds(mode):
    if mode == "random":
        return list(StratifiedKFold(NF, shuffle=True, random_state=SEED).split(y, (y>=2).astype(int)))
    return list(GroupKFold(NF).split(y, y, groups=groups))

def oof_xgb(X, mode):
    o = np.full(len(y), np.nan)
    for tr, te in folds(mode):
        m = xgb.XGBRegressor(**params); m.fit(X[tr], y[tr]); o[te] = m.predict(X[te])
    return o

def money(scores, T, K=500):
    kl = y[np.argsort(-scores)[:K]]; ev = int(kl[kl>=T].sum())
    return ev, ev*COST*EFF

def spear(s): return float(stats.spearmanr(s, y).correlation)

print("========== CORRECTED SET - model vs baseline (city = $0 here) ==========")
print(f"{'baseline':22s} {'ref':7s} rho={spear(base):+.4f} | >=1 ${money(base,1)[1]/1e6:.1f}M({money(base,1)[0]}ev) | >=2 ${money(base,2)[1]/1e6:.1f}M\n")
results = {}
oof_store = {}
for sn, cols in SETS.items():
    X = corr[cols].fillna(0.0).values.astype(float)
    for mode in ["random", "spatial"]:
        mo = oof_xgb(X, mode); oof_store[(sn,mode)] = mo
        line = f"{sn:22s} {mode:7s} rho={spear(mo):+.4f} "
        row = {"spearman": round(spear(mo),4), "n_features": len(cols)}
        for T in (1,2):
            me, md = money(mo, T); be, bd = money(base, T)
            line += f"| >={T}: ${md/1e6:5.1f}M({me}ev) vs base {(md-bd)/1e6:+.1f}M "
            row |= {f"ge{T}_model_$M": round(md/1e6,1), f"ge{T}_vs_baseline_$M": round((md-bd)/1e6,1)}
        results[f"{sn}__{mode}"] = row
        print(line)
    print()

json.dump(results, open("corrected_retrain_results.json","w"), indent=2)

# deliverable: the list the City ignores, ranked by the BEST model (E, spatial OOF)
corr["model_score"] = oof_store[("E_crash_infra_spatial","spatial")]
(corr.sort_values("model_score", ascending=False).head(500)
     [["intersection_id","crashes_feat","KSI_label","model_score"]
      + [c for c in ("lon","lat") if c in corr.columns]]
     .to_csv("city_ignored_top500.csv", index=False))
print("Saved: city_ignored_top500.csv (ranked by crash+infra+spatial) + corrected_retrain_results.json")
try:
    files.download("city_ignored_top500.csv")
except Exception:
    pass


## How to read it

Three model rows (**A** crash-only, **D** +infra, **E** +infra+spatial), each on `random`
and `spatial` OOF, versus the persistence `baseline`.

- **`>=1: $XM(Nev) vs base +/-Y.YM`** = prevented harm from that model's top-500, and the
  gap vs the baseline. Positive = beats the no-ML heuristic.
- Watch the ladder **A -> D -> E**: if `rho` and the dollars climb and E's `vs base` is
  clearly positive on **both** splits, the spatial features are the win — ship **E**.
- Every model dollar is **extra vs the City** (it catches $0 on these <5-crash sites).
- `city_ignored_top500.csv` downloads automatically — the deliverable, now ranked by the
  best (E) model.

If E doesn't clearly beat D, the spatial signal isn't surviving the honest OOF/spatial split
despite its strong univariate AUC — tell me and we'll look at why (likely spatial
autocorrelation being correctly penalized).
